# NBA data: fetch, parse, assemble

Pulls twenty-one seasons of results from the official NBA stats API and writes
one file at `data/processed/games.csv`.

The output is committed, so this notebook does not need running. It is here to
show how the dataset was built and to rebuild it if a season is added.

The pipeline is in three stages, kept separate on purpose. **Fetching** talks to
the network and does nothing else, caching raw responses per season so a parsing
change never costs another download. **Parsing** turns the raw team-level rows
into one row per game. **Assembling** runs both across every season and writes
the result.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Walk up until the project root is found. Keying on the engine package rather
# than on a data folder matters: notebooks can accumulate their own data
# directory, which would stop the walk one level too early and leave the engine
# package off the import path.
ROOT = Path.cwd()
while not (ROOT / "engine").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = ROOT / "data"
RAW_DIR = DATA / "raw"
PROCESSED_DIR = DATA / "processed"

## The pipeline

In [ ]:
"""
Downloads NBA schedule and results from the official NBA stats API.

The API returns one row per team per game, so every game appears twice; the
parsing step collapses those pairs into one row per game.

Season labelling: "2023-24" in the API is the season ending in 2024.
"""

import time

from nba_api.stats.endpoints import leaguegamefinder

REQUEST_DELAY = 2.0

# Season types are fetched separately because the API treats them as distinct
# queries. Play-in games are returned under their own type from 2020-21 onward.
SEASON_TYPES = ["Regular Season", "Playoffs", "PlayIn"]


def season_label(end_year: int) -> str:
    """2024 -> '2023-24', matching the API's season string format."""
    return f"{end_year - 1}-{str(end_year)[-2:]}"


def fetch_season(end_year: int, overwrite: bool = False) -> pd.DataFrame:
    """
    Pull every game for one season, caching the raw response to disk.

    Fetching is separated from parsing because it is the slow, failure-prone
    half: it talks to the network, it is rate limited, and it should not have
    to be repeated every time the parsing changes.
    """
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    cache_path = RAW_DIR / f"nba_{end_year}_teamrows.csv"

    if cache_path.exists() and not overwrite:
        print(f"  {end_year}: cached")
        return pd.read_csv(cache_path, parse_dates=["GAME_DATE"])

    frames = []
    for season_type in SEASON_TYPES:
        try:
            finder = leaguegamefinder.LeagueGameFinder(
                season_nullable=season_label(end_year),
                league_id_nullable="00",          # 00 is the NBA
                season_type_nullable=season_type,
                timeout=60,
            )
            frame = finder.get_data_frames()[0]
        except Exception as exc:
            # PlayIn does not exist before 2020-21 and some types return errors
            # rather than empty frames. Not fatal.
            print(f"  {end_year} {season_type}: {type(exc).__name__}")
            time.sleep(REQUEST_DELAY)
            continue

        if len(frame):
            frame["season_type"] = season_type
            frames.append(frame)
            print(f"  {end_year} {season_type}: {len(frame)} team-rows")

        time.sleep(REQUEST_DELAY)

    if not frames:
        raise RuntimeError(f"No data returned for {season_label(end_year)}")

    team_rows = pd.concat(frames, ignore_index=True)
    team_rows["GAME_DATE"] = pd.to_datetime(team_rows["GAME_DATE"])
    team_rows.to_csv(cache_path, index=False)
    return team_rows


def collapse_to_games(team_rows: pd.DataFrame, end_year: int) -> pd.DataFrame:
    """
    Fold two team-level rows into one game-level row.

    MATCHUP normally reads from the perspective of the row's own team, so a
    game produces one 'BOS vs. MIA' row and one 'MIA @ BOS' row. Neutral-site
    games break that: the API hands both teams the identical '@' string, so
    filtering on the separator alone picks up both rows and the merge collapses.

    Instead, parse the matchup into its nominal away and home sides and assign
    each row by its own abbreviation. That is robust either way, and the games
    where no 'vs.' variant exists are exactly the neutral-site ones, such as the
    Mexico City game, the two Paris games and the NBA Cup finals in Las Vegas.
    Those are flagged rather than dropped: they count for the standings but
    carry no home advantage.
    """
    rows = team_rows.copy()

    parsed = rows["MATCHUP"].str.extract(
        r"^(?P<first>[A-Z]{3})\s+(?P<sep>vs\.|@)\s+(?P<second>[A-Z]{3})$"
    )
    if parsed["sep"].isna().any():
        bad = rows.loc[parsed["sep"].isna(), "MATCHUP"].unique()[:5]
        raise ValueError(f"Unparseable MATCHUP values: {bad}")

    # 'A vs. B' means A hosts; 'A @ B' means B hosts.
    is_vs = parsed["sep"].eq("vs.")
    home_abbr = parsed["first"].where(is_vs, parsed["second"])
    away_abbr = parsed["second"].where(is_vs, parsed["first"])

    rows["nominal_home"] = home_abbr
    rows["nominal_away"] = away_abbr
    rows["side"] = pd.NA
    rows.loc[rows["TEAM_ABBREVIATION"] == rows["nominal_home"], "side"] = "home"
    rows.loc[rows["TEAM_ABBREVIATION"] == rows["nominal_away"], "side"] = "away"

    if rows["side"].isna().any():
        bad = rows.loc[rows["side"].isna(), ["GAME_ID", "TEAM_ABBREVIATION", "MATCHUP"]]
        raise ValueError(f"Rows whose team is absent from their own matchup:\n{bad.head()}")

    # A game with no 'vs.' row anywhere is played at neither team's arena.
    has_vs = rows.groupby("GAME_ID")["MATCHUP"].transform(
        lambda s: s.str.contains("vs.", regex=False).any()
    )
    rows["neutral_site"] = ~has_vs

    home = rows[rows["side"] == "home"].rename(columns={
        "TEAM_ABBREVIATION": "home_team",
        "TEAM_NAME": "home_team_name",
        "PTS": "home_points",
    })
    away = rows[rows["side"] == "away"].rename(columns={
        "TEAM_ABBREVIATION": "away_team",
        "TEAM_NAME": "away_team_name",
        "PTS": "away_points",
    })

    games = home[[
        "GAME_ID", "GAME_DATE", "home_team", "home_team_name",
        "home_points", "season_type", "neutral_site",
    ]].merge(
        away[["GAME_ID", "away_team", "away_team_name", "away_points"]],
        on="GAME_ID",
        how="inner",
        validate="one_to_one",
    )

    games = games.rename(columns={"GAME_DATE": "date", "GAME_ID": "game_id"})
    games["season"] = end_year
    games["home_margin"] = games["home_points"] - games["away_points"]
    games["home_win"] = games["home_margin"] > 0
    games["is_playoff"] = games["season_type"] == "Playoffs"
    games["is_playin"] = games["season_type"] == "PlayIn"

    # Overtime is not reported directly, but minutes played are: a regulation
    # game is five players times forty-eight minutes, so 240 team-minutes, and
    # anything beyond that means extra periods. Per-player rounding scatters the
    # regulation total between roughly 237 and 244, and an overtime adds 25,
    # producing a second cluster around 265 with nothing in between. A cutoff at
    # 250 separates them cleanly. This matters because a game that went to
    # overtime was level at the end of regulation, and that tie is informative.
    if "MIN" in home.columns:
        minutes = home.set_index("GAME_ID")["MIN"]
        games["overtime"] = games["game_id"].map(minutes) > 250
    else:
        games["overtime"] = False

    columns = [
        "season", "game_id", "date", "season_type", "neutral_site",
        "away_team", "home_team", "away_points", "home_points",
        "home_margin", "home_win", "overtime", "is_playoff", "is_playin",
    ]
    return games[columns].sort_values(["date", "home_team"]).reset_index(drop=True)


def build_dataset(season_end_years) -> pd.DataFrame:
    """Run fetch and parse across every season and write one file."""
    all_seasons = []
    for year in season_end_years:
        print(f"NBA {season_label(year)}")
        team_rows = fetch_season(year)
        season_games = collapse_to_games(team_rows, year)
        all_seasons.append(season_games)
        print(f"  -> {len(season_games)} games")

    combined = pd.concat(all_seasons, ignore_index=True)

    # Seasons whose schedule or venue was not normal. Kept in the dataset, but
    # excluded anywhere a standard season is assumed, above all when calibrating
    # home advantage, which was absent in the 2020 bubble and distorted by empty
    # arenas in 2021.
    #   2012 - lockout, 66 games per team
    #   2020 - COVID suspension and bubble restart, uneven game counts
    #   2021 - COVID, 72-game schedule, limited attendance
    ATYPICAL_SEASONS = {2012, 2020, 2021}
    combined["atypical_season"] = combined["season"].isin(ATYPICAL_SEASONS)

    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    out_path = PROCESSED_DIR / "games.csv"
    combined.to_csv(out_path, index=False)
    print(f"\nWrote {len(combined)} games to {out_path}")
    return combined

## Build

In [ ]:
# 2006 through 2026. The league has had thirty teams in two conferences since
# 2005, so earlier seasons would be structurally different, and 2005 itself is
# dropped because the API returns only its postseason.
SEASONS = range(2006, 2027)

games = build_dataset(SEASONS)

## Data quality checks

What these are looking for: seasons that came back short or incomplete, franchise
codes that change between seasons and would break a prior built from last year,
and missing values anywhere.

In [ ]:
games = pd.read_csv(PROCESSED_DIR / "games.csv", parse_dates=["date"])

# Coverage: what is in the dataset, season by season.
coverage = games.groupby(["season", "season_type"]).size().unstack(fill_value=0)
coverage["total"] = coverage.sum(axis=1)
coverage["teams"] = games.groupby("season")["home_team"].nunique()
coverage["neutral"] = games.groupby("season")["neutral_site"].sum()
coverage["overtime"] = games.groupby("season")["overtime"].mean().round(3)
print(coverage.to_string())

# Franchise codes that do not appear in every season. Relocations and rebrands
# change the abbreviation, which makes one franchise look like two teams to a
# prior built from last season's ratings.
codes_by_season = games.groupby("season").apply(
    lambda g: set(g["home_team"]) | set(g["away_team"]), include_groups=False
)
all_codes = set().union(*codes_by_season)
print("\nfranchise codes not present in every season:")
for code in sorted(all_codes):
    present = [s for s in codes_by_season.index if code in codes_by_season[s]]
    if len(present) < len(codes_by_season):
        print(f"  {code}: {present[0]}-{present[-1]} ({len(present)} seasons)")

# Games per team: 82 in a normal season, fewer in the three atypical ones.
per_team = (
    pd.concat([
        games[games.season_type == "Regular Season"][["season", "home_team"]]
            .rename(columns={"home_team": "team"}),
        games[games.season_type == "Regular Season"][["season", "away_team"]]
            .rename(columns={"away_team": "team"}),
    ])
    .groupby(["season", "team"]).size()
    .groupby("season").agg(["min", "max"])
)
print("\ngames per team:")
print(per_team.to_string())

print("\nmissing values:")
print(games.isna().sum()[lambda s: s > 0])

## What the data looks like

Home advantage has declined steadily over the sample, and the spread of margins
has widened as scoring has risen. Both are picked up automatically, since both
are re-estimated on every run.

In [ ]:
reg = games[(games.season_type == "Regular Season") & (~games.atypical_season)]
regular_all = games[games.season_type == "Regular Season"]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Home advantage over time, the parameter that is visibly not stationary.
home_rate = regular_all.groupby("season")["home_win"].mean()
axes[0, 0].plot(home_rate.index, home_rate.values, marker="o")
axes[0, 0].axhline(0.5, color="grey", linestyle="--", linewidth=1)
axes[0, 0].set_title("Home win rate by season")
axes[0, 0].set_ylabel("share of home wins")

# Margin distribution, the shape the normal approximation leans on.
axes[0, 1].hist(reg["home_margin"], bins=60, edgecolor="none")
axes[0, 1].axvline(0, color="grey", linestyle="--", linewidth=1)
axes[0, 1].set_title(
    f"Home margin, mean {reg['home_margin'].mean():.2f}, sd {reg['home_margin'].std():.2f}"
)
axes[0, 1].set_xlabel("home points minus away points")

# The same home advantage measured in points rather than wins.
margin_by_season = regular_all.groupby("season")["home_margin"].mean()
axes[1, 0].plot(margin_by_season.index, margin_by_season.values, marker="o")
axes[1, 0].axhline(0, color="grey", linestyle="--", linewidth=1)
axes[1, 0].set_title("Mean home margin by season (points)")

# Scoring level, context for why margin variance has grown.
total_points = regular_all.assign(
    total=lambda d: d.home_points + d.away_points
).groupby("season")["total"].mean()
axes[1, 1].plot(total_points.index, total_points.values, marker="o")
axes[1, 1].set_title("Mean combined points per game")

plt.tight_layout()
plt.show()